<p><small><small>This Notebook is made available subject to the licence and terms set out in the <a href = "http://www.github.com/google-deepmind/ai-foundations">AI Research Foundations Github README file</a>.</small></small></p>

<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C7-white-bg.png">

# Lab: Train and Evaluate your Conversational Model

<a href='https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_8/gdm_lab_8_9_conversation_part_2.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

3-4 hours of activities and 1-3 hours of waiting for models to train

In this second lab, you will train and evaluate your conversational model.

Similarly, as in the previous activities, this lab also contains a worked example for every step. Consider the code for the worked example as a template for your own implementation. Given that the steps for fine-tuning a conversational model do not vary too much, you may be able to run a lot of the existing code without changing it too much. However, some steps may still not be applicable to your project. You may have to complete additional steps, so treat the template in this lab just as a starting point and feel free to add or delete code cells to this notebook as you see fit.


## Overview

In this second part of implementing your capstone project, you will focus on fine-tuning and evaluating your conversational model.

### What you will learn

By the end of this second part of the capstone project, you will be able to:

* Fine-tune a large language model using LoRA to be used as a conversational model.

* Perform systematic manual evaluation of model responses.

### Tasks

In this lab, you will:

* Load the preprocessed dataset that you prepared in the first part.

* Configure the Gemma model and LoRA hyperparameters.

* Execute the fine-tuning process.

* Test the fine-tuned model and run manual spot checks on unseen data.

* Iterate on the hyperparameter settings and fine-tuning process until you are satisfied with your model's behavior.

The end result of this lab will be a conversational model that can respond to user queries as specified in your task description.

### What you will use

This lab builds on materials from several previous courses. If you would like to refresh your knowledge on any of the following topics, you can find the relevant materials in the following courses:

- **General knowledge on LLMs and transformers** (Courses 01 Build Your Own Small Language Model, 03 Design and Train Neural Networks, and 04 Discover the Transformer Architecture).

- **Fine-tuning models using LoRA** (Courses 05 Fine-tune Your Model and 07 Accelerate Your Model).  


**This lab needs to be run on GPU. Choose a T4 GPU.** See the section "How to use Google Colaboratory (Colab)" below for instructions on how to do this.

## How to use Google Colaboratory (Colab)

Google Colaboratory (also known as Google Colab) is a platform that allows you to run Python code in your browser. The code is written in cells that are executed on a remote server.

To run a cell, hover over a cell, and click on the `run` button to its left. The run button is the circle with the triangle (▶). Alternatively, you can also click on a cell and use the keyboard combination Ctrl+Return (or ⌘+Return if you are using a Mac).

To try this out, run the following cell. This should print today's day of the week below it.

In [ ]:
from datetime import datetime
print(f"Today is {datetime.today():%A}.")

Note that the order in which you run the cells matters. When you are working through a lab, make sure to always run all cells in order. Otherwise, the code might not work. If you take a break while working on a lab, Colab may disconnect you and in that case, you have to execute all cells again before  continuing your work. To make this easier, you can select the cell you are currently working on and then choose __Runtime → Run before__  from the menu above (or use the keyboard combination Ctrl/⌘ + F8). This will re-execute all cells before the current one.

### Using Colab with a GPU

Follow these steps to run the activities in this lab on a GPU:

1.  In the top menu bar, click on **Runtime**.
2.  Select **Change runtime type** from the dropdown menu.
3.  In the pop-up window under **Hardware Accelerator**, select **GPU** (usually listed as `T4 GPU`).
5.  Click **Save**.

Your Colab session will now restart with GPU access.

Note that access to GPUs is limited and at times, you may not be able to run this lab on a GPU. All activities will still work but they will run slower and you will have to wait longer for some of the cells to finish running.


## Imports

The following cell contains imports for a range of libraries that you may use for fine-tuning and evaluating your model. Depending on your implementation, you may need to import additional libraries.

In [ ]:
# Standard library imports.
import io # Input/output operations.
import json # JSON data handling.
import os # Operating system interface.
import random # For random number generation and shuffling of data.
import re # Regular expressions for text processing.
import unicodedata # Unicode character database.
from textwrap import fill  # Text wrapping utilities.
from typing import Any, Dict, Tuple, List

# Third-party data and utility libraries.
import numpy as np # Numerical computing.
import pandas as pd # Data manipulation and analysis.
from tqdm import tqdm # Progress bars.

import matplotlib.pyplot as plt # Plotting and visualisation.

# Set Keras and JAX settings.
os.environ["KERAS_BACKEND"] = "jax"

# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

# Deep learning and AI libraries.
import jax.numpy as jnp # JAX numerical computing.
import keras # High-level neural networks API.
import keras_nlp # Keras NLP extensions.

# Google Colab specific.
from google.colab import drive # Access to Google Drive.
from google.colab import userdata # Access to Colab secrets.

##  Activity 1: Load your data

As a first step, you will load your training and test data that you prepared in the previous activity from Google Drive. See the worked example for how to do this if you followed the format in the previous lab.

In [ ]:
# Connect this notebook to Google Drive.
drive.mount('/content/drive')

#### Worked example: Load data

```python
train_df = pd.read_json(
    "/content/drive/MyDrive/conversational_training_data.jsonl", lines=True
)
test_df = pd.read_json(
    "/content/drive/MyDrive/conversational_test_data.jsonl", lines=True
)

# Turn dataframe into a list of dictionaries, as expected by the model
# fine-tuning code.
train_data = train_df.to_dict("records")
test_data =  test_df.to_dict("records")

# Print the first element in train_data and test_data for verifcation.
print(train_data[0])
print(test_data[0])
```

#### Your implementation

In [ ]:
# Add your code for loading your data here.


### Configure your Kaggle API key for Gemma

To access Gemma models, provide your Kaggle credentials. If you have not set your Kaggle API key yet take a look at the instruction in the following cells.

#### Set up a Kaggle account



To run this notebook, you will have to sign up for [Kaggle](https://www.kaggle.com), a platform that hosts datasets and models for machine learning, and sign the agreement for using the Gemma 3 model. This is required so that you can download the weights of the Gemma model for fine-tuning.

**Step 1: Create your Kaggle account**

* Go to the Kaggle website: https://www.kaggle.com

* Click the "Register" button in the top-right corner.

* You can sign up using your Google account (recommended for easy Colab integration) or by entering an email and password.

* Follow the on-screen prompts to complete your registration and verify your email.

**Step 2: Sign the Gemma 3 model agreement**

* Make sure you are logged into your new Kaggle account.

* Go directly to the Gemma 3 model card page: https://www.kaggle.com/models/keras/gemma3/keras/

* You should see a "Request Access" button.

* Click the button, read through the license agreement, and click "Accept" to gain access to the model. You must do this before the API will let you download the model.

**Step 3: Generate your Kaggle API key**

* From any Kaggle page, click on your profile picture or icon in the top-right corner.

* Select "Account" from the drop-down menu.

* Scroll down to the "API" section.

* Click the "Create Legacy API Key" button in the "Legacy API Credentials" section.

* This will immediately download a file named `kaggle.json` to your computer. This file contains your username and your secret API key. Keep it safe.

**Step 4: Set your API Key in Colab**

* Click the "key" icon 🔑 in the left-hand sidebar.

* You will see the "Secrets" panel.

* Now, open the kaggle.json file you downloaded on your computer. It's a simple text file and will look like this:

   ```json
   {"username":"YOUR_KAGGLE_USERNAME","key":"YOUR_KAGGLE_API_KEY"}
   ```
* Note: Paste the values exactly as they appear inside the quotation marks in your kaggle.json file, but do not include the quotation marks themselves in the Colab value field.

* In the Colab Secrets panel, create two new secrets:

   1. Name: `KAGGLE_USERNAME`

      Value: Copy and paste `YOUR_KAGGLE_USERNAME` from your `kaggle.json` file.

   2. Name: `KAGGLE_KEY`

      Value: Copy and paste `YOUR_KAGGLE_API_KEY` from your `kaggle.json` file.

* For both secrets, make sure the "Notebook access" toggle is switched on.


#### Load the Kaggle username and key

In [ ]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

## Activity 2: Preparation for fine-tuning


### Prepare data for Gemma3 LoRA fine-tuning

Before starting fine-tuning, ensure that your training data is formatted correctly for the specific model you plan to use. In the previous notebook, you created JSONL files containing instruction-tuning data, which can be reused across different models and fine-tuning methods.

However, each model has its own formatting requirements. The Gemma models support only two conversational roles, **user** and **model**, but not a **system** role. In other words, prompts and responses are structured as direct exchanges between the user and the model, without a separate system message to provide global instructions or context (such as defining tone, style, or constraints). Higher-capacity LLMs, like Gemini, also use a system role for this purpose, allowing developers to set overall behavior at the start of a conversation. Because Gemma does not support the system role, each training example must explicitly include any necessary context or instruction within the user prompt itself. For more information, see the [Gemma formatting and system instructions document](https://ai.google.dev/gemma/docs/core/prompt-structure/).
  


------
> 💻 **Your task:**
>
> 1. **Implement conversation formatting**:
>    - **Special tokens**: The worked example uses `<start_of_turn>` and `<end_of_turn>`.
>    - **Role labels**: The worked example uses `user` and `model` roles.
>    - **Turn structure**: Adapt the `format_io_as_turns()` function in the worked example to ensure it matches your conversation format requirements.
>
> 2. **Verify formatted output**:
>    - Write code to examine examples to verify:
>      - User turns are properly formatted with questions or prompts.
>      - Model turns contain the expected conversational responses.
>      - Special tokens are correctly positioned.
>      - The final data dictionary has the required "prompts" and "responses" keys.
>
> 3. **Customize for your domain**:
>    - Ensure the formatting aligns with how you want the model to behave during inference.
>    - For multi-turn dialogue, additional changes will be required to include conversation history within each input so that the model can maintain context across exchanges.
>

------

To help Gemma3-1B clearly distinguish between the user prompt and the expected model reply, each part of the conversation is wrapped with special tokens such as `<start_of_turn>` and `<end_of_turn>`.






#### Worked example: Add special tokens





The code below demonstrates how to:

* Add the `<start_of_turn>` / `<end_of_turn>` tags to each prompt-response pair.
* Prepare a dictionary with `prompts` and `responses` keys, ready for training a Gemma3-1B Keras model.

This approach ensures that the model sees well-delimited Q&A pairs and can be fine-tuned.

```python
# Special tokens for start/end of a turn.
SOT = "<start_of_turn>"
EOT = "<end_of_turn>"


def format_io_as_turns(
    input_text: str, output_text: str, sot: str = SOT, eot: str = EOT
) -> tuple[str, str]:
    """Format a single input/output pair for LoRA training.

    Format a single input/output pair for LoRA training by wrapping the pair
    with <start_of_turn> and <end_of_turn> markers.

    Args:
      input_text: The full user prompt text. This typically includes the user
        prompt, e.g., "Can you tell me a popular dish from Eastern Africa?".
      output_text: A response to the user's query.
      sot: The token used to indicate the start of a turn. Defaults to
        "<start_of_turn>".
      eot: The token used to indicate the end of a turn. Defaults to
        "<end_of_turn>".

    Returns:
      formatted_q: The user turn string, e.g.
        "<start_of_turn>user\n<prompt><end_of_turn>".
      formatted_a: the model turn string, e.g.
        "<start_of_turn>model\n<label><end_of_turn>".
    """

    formatted_q = f"{sot}user\n{input_text}{eot}"
    formatted_a = f"{sot}model\n{output_text}{eot}"
    return formatted_q, formatted_a

inputs = []  # Model prompts.
targets = []  # Expected responses.

for example in train_data:
    # Format each record into a user/model dialogue pair.
    q, a = format_io_as_turns(example["input"], example["output"])
    inputs.append(q)
    targets.append(a)

# Show the first set of input and output.
print(inputs[0])
print(targets[0])

# The Gemma3 Keras models require the data as dictionaries with keys
# "prompts" for the inputs and "responses" for the outputs.
data = {"prompts": inputs, "responses": targets}
```

#### Your implementation

In [ ]:
# Add your code to add special delimiter tokens here.


### Load Gemma using Keras
Once your data is ready, you now need to fine-tune your Gemma model using LoRA,  as you familiarized yourself with in 05 Fine-tune Your Model. You will train only a small number of additional parameters whilst keeping the base model frozen. You may find additional information in [Google's *Fine-tune Gemma in Keras using LoRA* documentation](https://ai.google.dev/gemma/docs/core/lora_tuning).

------
> 💡 **Tip: Curious about model size and performance trade-offs?**
>
> You can experiment with the larger **Gemma3-4B** model to compare accuracy and output quality against the 1B version.  
> Keep in mind that Colab's default **T4 GPU** may not have enough memory to fine-tune the 4B model in full precision.  
> To make this feasible, apply the **optimization techniques** introduced in Course 07 Accelerate Your Model, such as mixed-precision inference, parameter-efficient fine-tuning, or low-bit quantisation, to reduce memory usage and stay within hardware limits.
>
------
More information about the Gemma3 models can be found in [Google's *Gemma3 model overview* document](https://ai.google.dev/gemma/docs/core).


#### Worked example: Load Gemma3



```python
# Load the Gemma3-1B Keras model.
model = keras_nlp.models.Gemma3CausalLM.from_preset("gemma3_1b")
model.summary()
```

#### Your implementation

In [ ]:
keras.utils.set_random_seed(812)  # Replicate randomness in Keras.

# Add your code for loading a Gemma3 model here.


------
> 💡 **Tip: Test the base model before fine-tuning**
>
> Before starting LoRA fine-tuning, **test the base Gemma model** with a few example prompts.
> This helps establish a baseline for how the unadapted model responds in conversational tasks (e.g., answering food-related questions about Africa).
> You can reuse these same prompts after fine-tuning to directly compare how LoRA improves the model's contextual accuracy, tone, and relevance.
>
> For efficiency, **load the model only once** and keep it in memory for both the pre-fine-tuning and post-fine-tuning evaluations.
> On Google Colab, where the default **T4 GPU has limited memory**, repeatedly loading different versions of the model may exceed resources and cause out-of-memory errors.
------

### Evaluate the base model's performance

Before fine-tuning, this is a good moment to **test the base (foundation) Gemma model** with a small set of example prompts from the training data.
You can **reuse these same prompts after fine-tuning** to see how LoRA adaptation changes the model's conversational responses, giving you a direct, side-by-side comparison of performance.
For efficiency, **load the model only once** and keep it in memory while you carry out both the pre-fine-tuning tests and the later evaluation.
On Google Colab, where the default **T4 GPU has limited memory**, repeatedly loading different versions of the model can easily exceed available resources and trigger out-of-memory errors.

The code below shuffles the test data (some of the prompt-output pairs that you constructed in the previous notebook) and generates a response for the first five examples in your test set.

#### Worked example: Evaluate the base model's performance

```python
# Shuffle the data.
random.shuffle(test_data)

print("Testing baseline prompts on the base model:\n")

# Test the base model on a shuffled sample.
for entry in test_data[:5]:
    prompt = entry["input"]
    expected_response = entry["output"]
    print(f"Prompt: {prompt}")
    print(f"Sample of an expected response: {expected_response[:100]}...")
    print(f"Base model response: {model.generate(prompt, max_length=450)}")
    print("-" * 50)
```


#### Your implementation

In [ ]:
# Add code for evaluating the base model's performance here.


------
> ⚠️ **Understanding base model limitations**
>
> You will likely notice that the base Gemma3-1B model produces inconsistent or off-topic responses when asked to answer conversational prompts about African food. This behavior is expected, and below is explained why:
>
> **Base models vs instruction-tuned models**:
>
> The Gemma3-1B model used here is a **base (pretrained) model**, not an **instruction-tuned** variant. As you have also experienced in Course 05 Fine-Tune Your Model, base models are trained solely on **next-token prediction**, learning to predict the most likely next word given the preceding context. They have not been further trained to interpret and follow human instructions.
>
> In contrast, **instruction-tuned models** (such as `gemma3_instruct_1b`) undergo additional training on instruction-response pairs, teaching them to interpret prompts as commands and respond appropriately. When you ask a base model a conversational question, it may simply continue generating text in a way that seems plausible given the input, rather than providing a helpful, contextually relevant answer.
>
> For more information about the model variants, see [Google's *Run Gemma content generation and inferences* documentation](https://ai.google.dev/gemma/docs/run).
>
> **Why small models struggle more**:
>
> With only 1 billion parameters, the Gemma3-1B model has limited capacity to capture complex patterns and implicit task instructions from context alone. Larger models can sometimes infer task requirements from prompt phrasing, but smaller base models are more prone to producing off-topic or hallucinated responses, generating text that sounds plausible but is factually incorrect or irrelevant to the question asked.
>
> **Why use a base model for fine-tuning**:
>
> Despite these limitations, base models are often preferred for fine-tuning. Because they have not been optimized for a specific instruction-following format, they offer a more neutral starting point. This allows LoRA fine-tuning to more effectively reshape the model's behavior toward your specific conversational style and domain knowledge, without having to override patterns learned during instruction tuning.
>
> By comparing the base model's outputs before and after fine-tuning, you will see a clear demonstration of how LoRA adaptation transforms an unreliable base model into a capable, domain-specific conversational model.
------


## Activity 3: Activate and run LoRA


------
> 💻 **Your task:**
>
> Set up your hyperparameters and fine-tune your Gemma model using LoRA:
>
> 1. **Experiment with LoRA rank**: Start with `rank=4` (faster), `rank=6` (medium) or try `rank=8` (potentially better quality, but requires more memory):
>    ```python
>    lora_rank = 6
>    model.backbone.enable_lora(rank=lora_rank)
>    ```
>
> 2. **Adjust training parameters**:
>    - **Learning rate**: A good value to start with is `5e-5` but depending on your dataset size and how diverse your data is, you likely also want to experiment with different learning rates.
>    - **Number of epochs**: For the worked example, it is set to 3. Dialogue-based models may typically require more epochs than other type of models such as classification because they learn free-form response generation and conversational structure rather than closed-set label prediction. Training progress should be monitored to avoid overfitting.
>    - **Batch size**: Limited memory on T4 GPUs might prevent you from increasing this parameter. If you have access to a GPU with more memory, you can raise this to speed up and stabilize training.
>    - **Sequence length**: Adjust based on your text length (default: 450). You might encounter out of memory problems on a T4 GPU if sequence length is too long during fine-tuning.
>
> 3. **Monitor training progress**:
>    - Watch the loss decrease over epochs.
>    - Check sparse categorical accuracy improvement.
>    - Stop early if the model starts overfitting.
>    - You may also implement code that prints a prompt at every step.
>
> 4. **Iterate as needed**: If results are not satisfactory:
>    - Try different hyperparameters.
>    - Increase/decrease epochs.
>    - Adjust the LoRA rank.
>
------


#### Worked example: Set up hyperparameters


```python
# Set the LoRA rank.
lora_rank = 6
# Set the learning rate.
learning_rate = 5e-5
# Determine the number of epochs.
num_epochs = 3
# Set the batch size.
batch_size = 1
# Set the maximum length.
sequence_length = 450

# Activate LoRA.
model.backbone.enable_lora(rank=lora_rank)

# Check learning rate.
print(model.optimizer.learning_rate)
```

#### Your implementation

In [ ]:
# Add your code to set the hyperparameters and enable LoRA training.


-----
> **ℹ️ Info: Understanding LoRA hyperparameters**
>
> The following parameters control how LoRA fine-tuning adapts the model to your conversational task:
>
> **`lora_rank`** (integer, e.g., 4, 8, 16):
> The rank of the low-rank matrices added to the model's attention layers. Higher ranks capture more complex patterns but increase memory usage and training time. For small datasets or simple conversational tasks, `rank=4` is often sufficient. For larger datasets or tasks requiring more nuanced proessing, try `rank=8` or higher.
>
> **`learning_rate`** (float, e.g., 1e-5, 5e-5):
> Controls how much the model weights are adjusted during each training step. Higher rates learn faster but risk overshooting optimal values. Lower rates are more stable but train slower. Start with `2e-5` or `5e-5` for LoRA fine-tuning. If the model fails to learn (high loss), try increasing slightly. If training is unstable (loss fluctuates), reduce the rate.
>
> **`num_epochs`** (integer, e.g., 2, 5, 10):
> The number of complete passes through the training dataset. More epochs allow the model to learn more thoroughly, but too many can cause **overfitting** (memorising rather than generalizing). Start with 2-3 epochs for small datasets and increase if validation performance is still improving (monitor the `loss` and `sparse_categorical_accuracy` values).
>
> **`batch_size`** (integer, e.g., 1, 4, 8):
> The number of training examples processed together before updating weights. Larger batches provide more stable gradient estimates but require more GPU memory. On T4 GPUs with limited memory, `batch_size=1` is often necessary for large models like Gemma. If memory permits, increasing batch size can speed up training.
>
> **`sequence_length`** (integer, e.g., 256, 450, 512):
> The maximum number of tokens the model processes per input. Must be long enough to contain your full prompts and expected responses. Longer sequences require more memory. Analyze your training data to determine appropriate length and set it slightly above your longest expected input/output combination.
>
-----


-----
> **💡 Tip: Training data quality and why prompt diversity matters**
>
> When preparing training data for LoRA fine-tuning, **prompt diversity** is an important factor that affects model quality, especially in production environments.
>
> **The problem with duplicate prompts**:
>
> If your training data contains multiple examples with **identical prompts but different expected responses**, this can create conflicting learning signals for the model. During training, the model attempts to learn a mapping from inputs to outputs. When the same input is associated with different outputs, the model receives contradictory gradient updates, which can lead to:
>
> - **Unstable training**: Loss may fluctuate as the model oscillates between conflicting targets.
> - **Averaged or generic responses**: The model may learn to produce a middle-ground response that partially matches multiple targets but fully satisfies none.
> - **Reduced confidence**: The model becomes uncertain about the correct response, potentially producing inconsistent outputs during inference.
>
> **Why this matters for production**:
>
> In production environments, you typically want deterministic, reliable behavior. A model trained on conflicting examples may produce unpredictable responses to the same prompt, making quality assurance difficult and user experience inconsistent.
>
> **Improving prompt and response diversity**:
>
> If your prompts or responses are not diverse enough, consider modifying the examples in your training data or adding more examples to your training set. Similarly as with hyperparameters, you may have to iterate a bit on how you generate and preprocess the dataset until you get good results.
>
> Usually, taking time to establish prompt uniqueness in your training data may result in a more reliable and consistent conversational model.
>
-----


### Fine-tuning
It is now time to perform fine-tuning with LoRA. You can leave the hyperparameters as default or change them. You may wish to reduce the number of epochs. In the first run watch the loss and the sparse categorical accuracy for each epoch and adjust accordingly to achieve the desired accuracy without overfitting the model.

The time required to complete LoRA fine-tuning depends on several factors, including the size of the dataset, the length of the instruction pairs, and hyperparameters such as the number of epochs and LoRA rank. Depending on these variables, the process can take anywhere from a few minutes to several hours. For the default example dataset, the estimated fine-tuning time is about 35 minutes.

#### Worked example: Run fine-tuning



```python
model.optimizer.learning_rate = learning_rate
model.preprocessor.sequence_length = sequence_length

# Using a for loop provides better control over each iteration if you want to
# add custom logic, but otherwise identical to setting epochs=num_epochs.
for i in jnp.arange(num_epochs):
    print("\n\nEpoch:" + str(i + 1) + "\n")
    model.fit(data, epochs=1, batch_size=batch_size, verbose=1)
    # Add code to print model generations for one or two prompts after every
    # epoch here.
```

#### Your implementation

In [ ]:
# Add your code to fine-tune the model.


Ideally, you now have a fine-tuned Gemma model trained with LoRA for conversational generation. The next step is to evaluate how effectively the model responds to real user prompts by testing its dialogue quality, factual accuracy, and overall conversational performance.

## Activity 4: Evaluation


------
> 💻 **Your task:**
>
> Evaluate your fine-tuned model through human assessment, comparing it against the base model and testing its ability to generalize to unseen prompts.
>
> You can likely again adapt the code from the worked example for this. This will likely involve the following steps.
>
> 1. **Configure model generation parameters**:
>    - **`max_length`**: Use a value slightly higher than during fine-tuning. In this project, truncation occurred twice, first during data processing (to shorten long descriptions), and again during LoRA fine-tuning (default set to 450 tokens). For inference, increase this to around 600 to allow the model to generate fuller responses.
>
> 2. **Create evaluation prompts**:
>    - **Option A: Use provided dataset splits** (if available): Some datasets already include predefined training, validation, and test splits. In that case, it is good practice to keep those original splits rather than re-splitting the data in order to keep your results directly comparable to other work using the same dataset.
>    - **Option B: Use your own test dataset** (if available): If you split your original dataset earlier, use entries from your test set to generate new prompts not seen during training.
>    - **Option C: Manual prompt creation**: Design at least 10 test questions mixing some that were and some that were not in your training data.
>    - **Prompt variety and deviation testing**: Start with familiar formats, then gradually deviate to test the model's limits. As inspiration, here are some examples prompt variations for the worked example.
>      - **Baseline prompts** (similar to training): 'Recommend a well-known local dish from Nigeria'.
>      - **Slight variations**: 'What's a popular dish from Kenya?'' / 'Tell me about food from Ghana'.
>      - **Regional queries**: 'Recommend food from West Africa' / 'What food should I try in Eastern Africa?'.
>      - **Format deviations**: 'I'm visiting Morocco, what local cuisine should I experience?'.
>      - **Edge cases**: 'What's the best African food?' / 'Pick a random Nigerian or Ghanaian food to try'.
>    - **Evaluation focus**: Test when the model maintains expected format vs. when it breaks down or responds unexpectedly.
>
> 3. **Compare base model vs. fine-tuned model outputs**:
>    - Run the same prompts through both the original base model and your fine-tuned model.
>    - Compare response quality, relevance, and domain-specific accuracy.
>    - Document how fine-tuning has improved (or changed) the model's behavior for your use case.
>
> 4. **Test generalization with unseen prompts**:
>    - Use prompts that differ from your training data to assess how well the model generalizes.
>    - Include novel phrasings, related but untrained topics, and edge cases.
>    - Evaluate whether the model applies learned patterns appropriately to new contexts.
>
> 5. **Run conversational evaluation**:
>    - Execute the code to generate model responses to your test prompts.
>    - Review each response for quality, accuracy, and appropriateness.
>    - Note patterns in helpful vs. problematic responses.
>
------


#### Worked example: Manual spot checks

The following sample code is a template on how to run **manual spot checks** on a fine-tuned conversational model.

What the worked example code does:
- **Prepares a small set of conversational prompts** about African food and culture.
- **Formats each prompt** into a conversation with `<start_of_turn>` and `<end_of_turn>` tokens and sends it to the fine-tuned model.
- **Displays the model's conversational responses** for quality evaluation.

```python
def generate_answer(
    question: str,
    sot: str = "<start_of_turn>",
    eot: str = "<end_of_turn>",
) -> str:
    """Generate a model answer for a single question. Each call is stateless.

    Args:
      question: The question text.
      sot: Start-of-turn token. Default: "<start_of_turn>".
      eot: End-of-turn token. Default: "<end_of_turn>".

    Returns:
      The model's output wrapped as a model turn, e.g.
        "<start_of_turn>model\n...<end_of_turn>".
    """
    # Build the complete user+model turn sequence
    prompt = (
        f"{sot}user\n"
        f"{question}{eot}\n"
        f"{sot}model\n"
    )

    raw_output = model.generate(prompt, max_length=600)

    # Extract only the newly generated content by replacing the prompt prefix
    completion = raw_output.replace(prompt, "").strip()
    
    # Clean up trailing end token if present
    if completion.endswith(eot):
      completion = completion[:-len(eot)].strip()

    # Return properly wrapped model response    
    return f"{sot}model\n{completion}{eot}"


def ask(question: str) -> str:
    """Generate and return a formatted model response for a given user question.

    Args:
      question: The question text.

    Returns:
      The formatted model response.
    """
    formatted = question.strip()
    return generate_answer(formatted)
```

After defining the necessary functions to format test examples, you can prompt the fine-tuned model.

```python
print(ask("Can you tell me a popular dish from Eastern Africa?"))
print(ask("I plan to visit South Africa. What food should I definitely try?"))
print(ask("List a popular food to try in Ghana."))
print(ask("What is a popular dish from Egypt?"))
print(ask("Can you suggest a good dish from Eastern Africa?"))
print(ask("Give me a signature food in Ethiopia."))
print(
    ask(
        "I need to cook Northern African food for a dinner party. "
        "Recommend a typical dish from there."
    )
)
```

#### Your implementation

In [ ]:
# Add code for perfoming spot checks here.


### Comparing base model vs fine-tuned model outputs

Now that the model has been fine-tuned with LoRA, you can test it using the **same prompts** you used earlier to evaluate the base model. This allows for a direct, side-by-side comparison of how fine-tuning has affected the model's conversational behavior.

Run the cell below to generate responses for the same set of prompts using the fine-tuned model. Compare the outputs with what you observed earlier from the base model:

- **Base model (before fine-tuning)**: Likely produced inconsistent, off-topic, or hallucinated responses because it was trained only on next-token prediction and did not understand the conversational task.

- **Fine-tuned model (after LoRA)**: Ideally this will now produce more coherent, contextually relevant responses that match the expected conversational style and domain knowledge, demonstrating how LoRA adaptation has taught the model to generate answers that are closer to the ones you included in your training data.


#### Worked example: Test fine-tuned model on training examples

```python
# Test the fine-tuned model on a shuffled sample from the
# training data compiled earlier (seen during training).

print("Testing model on seen prompts:\n")

for entry in train_data[:5]:
    prompt = entry["input"]
    expected_response = entry["output"]
    print(f"Prompt: {prompt}")
    print(f"Sample of an expected response: {expected_response[:100]}...")
    print(f"Fine-tuned model response: {generate_answer(prompt)}")
    print("-" * 50)
```

#### Your implementation

In [ ]:
# Add your code to test the fine-tuned model on seen examples.


### Testing generalization with unseen prompts

To better assess whether your fine-tuned model has truly learned the conversational task, you can test it with **new prompts** that were not part of the training data.



#### Worked example: Test fine-tuned model on test examples

The following code uses prompts from `test_data`, which should include prompts that the model has never seen during fine-tuning. If the model generalizes well, it should still produce coherent, contextually relevant responses for these novel examples.


```python
# Test the fine-tuned model with prompts from the test set
# (unseen during training).

print("Testing model on unseen prompts:\n")

for entry in test_data[:5]:
    prompt = entry["input"]
    expected_response = entry["output"]
    print(f"Prompt: {prompt}")
    print(f"Sample of an expected response: {expected_response[:100]}...")
    print(f"Fine-tuned model response: {generate_answer(prompt)}")
    print("-" * 50)
```

#### Your implementation

In [ ]:
# Add code to generate responses for unseen examples.


Ideally, your fine-tuned model demonstrates reasonable generalization to unseen examples but its performance may vary depending on how well the information required to respond is represented in the training data. If your model does not work very well, then increasing the amount of training data and/or the number of training epochs can further improve the accuracy and reliability of the fine-tuned model.

Crucially, **the model should now follows the expected response format in most cases**, producing structured conversational answers rather than the off-topic or rambling text generated by the base model earlier. This structured behavior is a direct result of LoRA fine-tuning, which taught the model:

1. **What task to perform**: For example, in the context of the worked example, to generate informative responses about African food and culture.
2. **How to respond**: Produce conversational answers following the learned style and format.

The `<start_of_turn>` and `<end_of_turn>` tokens play a key role here. These special delimiters clearly demarcate user input from model output, helping the fine-tuned model understand when to 'listen' and when to 'respond'. This turn-based structure, standard in instruction-tuned models, guides the model toward appropriate, on-task behavior.

------
> ⚠️ **Prompt sensitivity**
>
> Fine-tuned models, especially smaller ones like Gemma3-1B, can be sensitive to **variations in prompt wording**. A prompt phrased slightly differently from the training examples may produce noticeably different responses. For example, in the context of the worked example, asking 'What food should I try in Kenya?' versus 'Recommend a dish from Kenya' might yield different quality outputs.
>
> When evaluating your model, experiment with different phrasings of similar questions to assess how robust the model is to prompt variations. If you notice inconsistent behavior, consider diversifying the prompt templates in your training data to improve generalization.
------


-----
> **ℹ️ Info: Evaluating conversational models**
>
> Conversational models can be evaluated through a mix of automated metrics and human judgment, depending on how open-ended or task-focused the system is. **Human evaluation remains the gold standard for conversational models**, as it captures qualities like fluency, helpfulness, tone, and factual accuracy that automated metrics often miss. Human evaluators typically assess whether the model's replies are relevant, natural, and contextually appropriate.
>
> **Automated evaluation methods**, such as BLEU and ROUGE, measure the degree of word overlap between the model's output and reference responses, making them useful for factual or goal-oriented models (commonly used to evaluate machine translation). More advanced approaches like METEOR and BERTScore go beyond surface-level matching by accounting for synonyms or semantic similarity, offering a better reflection of meaning alignment. Neural-based metrics such as DialogRPT estimate how much a response aligns with human conversational preferences.
>
> For more sophisticated evaluation, **larger more capable LLMs** such as Gemini can also be used as evaluators. These models can score model responses on relevance, accuracy, or tone, acting as automated 'judges' that approximate human feedback.
>
> For specific applications, **behavioral checks** can also be applied, for example, measuring intent detection accuracy, named entity recognition, response diversity, or factual correctness. No single metric fully captures conversational quality, but combining automated scoring for basic consistency with selective human evaluation for nuance provides the most reliable way to assess model performance.
>
------

------
> 💡 **Tip: Try to semi-automate your evaluation, if possible**
>
> For conversational models, fully automated evaluation is often difficult because responses are open-ended and conversational. However, in structured cases, such as simple Q&A prompts like those used in the worked example, a partial automation may be possible via behavioral checks.
>
> A script could, for example, check whether the model's response mentions food originating from the correct region. This can be done through basic word search or keyword matching, similar to the method previously used to map foods to geographic areas.
>
>You can extract the generated response from between `<start_of_turn>model` and `<end_of_turn>` in the model_output column, then compare it against expected keywords for that location.
> While this approach won't capture conversational quality, it may assess factual consistency at scale.
>
------

#### Evaluating model responses: Spot-checking guide

After running the evaluation prompts, you will see the model's responses to various prompts. Conversational model evaluation focuses on conversational quality, factual accuracy, and response appropriateness rather than exact label matching.

To give you an idea how you may evaluate these responses, consider the following evaluation framework that has been designed for the worked example using African food data from the African Galore dataset.
It can be adapted to other domains or cultural contexts by revising the criteria to reflect the relevant subject matter, regional focus, and ethical considerations.

#### Understanding the model output format

The model generates conversational responses in a structured format. Here's an example of what you might see:

```
<start_of_turn>user
Recommend a well-known local dish from Nigeria.<end_of_turn>
<start_of_turn>model
This dish captures the authentic flavor and spirit of Nigeria in Africa. Pepper soup — Pepper soup is a hearty Nigerian soup made with a spicy, richly flavored stock of meat, vegetables, and sometimes fish.<end_of_turn>
```

**Key components to identify:**
- **User question**: The original query about African food/culture (e.g., 'Recommend a well-known local dish from Nigeria').
- **Model response**: The informative answer provided by the model.
- **Response structure**: Typically follows the pattern 'This dish captures the authentic flavor and spirit of [location]. [Food name] — [description]'.
- **Special tokens**: `<start_of_turn>`, `<end_of_turn>` (conversation delimiters that can be ignored during evaluation).

#### Step-by-step evaluation process

**1. Extract the model's response**
From the model output, identify the text between `<start_of_turn>model` and `<end_of_turn>`. This contains the complete conversational response.

**2. Assess response quality**
Unlike other pathways like classification, there's no single 'correct' answer. Instead, evaluate multiple dimensions:
- **Factual accuracy**: Is the information about the food/location correct?
- **Relevance**: Does the response directly address the user's question?
- **Cultural authenticity**: Does it demonstrate appropriate knowledge of African culture?
- **Conversational tone**: Is the response natural and engaging?

**3. Check geographic consistency**
- **Location matching**: Does the recommended food actually come from the requested region/country?
- **Cultural context**: Is the cultural information accurate and respectful?
- **Regional awareness**: Does the response show understanding of African geography and diversity?

#### Example evaluation

| User Question | Model Response | Quality Assessment |
|---------------|------------------|-------------------|
| 'Recommend a dish from Nigeria' | 'This dish captures the authentic flavor and spirit of Nigeria. Jollof Rice — A vibrant rice dish...' | ✅ **EXCELLENT**: Correct country, authentic dish, engaging description. |
| 'Tell me about food from Ghana' | 'This dish captures the authentic flavor and spirit of Nigeria. Banku — A fermented corn...' | ⚠️ **MIXED**: Correct dish for Ghana, but mentions Nigeria in location text. |
| 'What should I try in Eastern Africa?' | 'This captures the spirit of Africa. Pizza — An Italian dish...' | ❌ **POOR**: Wrong cuisine, no cultural relevance, generic location. |

#### What to look for during spot-checking

**✅ Excellent responses:**
- Accurate geographic and cultural information.
- Natural, conversational tone that matches training style.
- Specific food names with informative descriptions.
- Appropriate cultural context and respect.
- Consistent response formatting.

**✅ Good responses:**
- Generally accurate information with minor inconsistencies.
- Engaging tone but may lack some cultural depth.
- Correct food-location pairing with adequate descriptions.
- Maintains conversational flow.

**⚠️ Concerning responses:**
- Geographic mismatches (wrong country/region for the food).
- Overly generic or repetitive language.
- Cultural inaccuracies or insensitive content.
- Inconsistent formatting or incomplete responses.

**❌ Poor responses:**
- Completely incorrect or irrelevant information.
- Non-African foods recommended for African regions.
- Offensive or culturally inappropriate content.
- Broken formatting or incoherent responses.

#### Systematic evaluation framework

Creating a structured evaluation framework helps make model assessment consistent, transparent, and aligned with project goals. It allows evaluators to balance qualitative judgment with measurable criteria, making it easier to identify specific areas for improvement.

The weighting of categories is flexible and should be adapted to fit the focus of each project. You can assign greater importance to certain dimensions, such as factual accuracy, conversational quality, or cultural sensitivity, depending on their model's purpose and intended users.

Below is an indicative framework for systematic evaluation for the worked example.

**1. Content accuracy**
- Geographic correctness: Does the food match the requested location?
- Cultural authenticity: Is the cultural information accurate and respectful?
- Factual details: Are the food descriptions accurate?

**2. Conversational quality**
- Natural flow: Does the response sound conversational and engaging?
- Tone consistency: Does it match the helpful, informative style from training?
- Completeness: Does it fully address the user's question?

**3. Response structure**
- Format consistency: Does it follow the expected 'This dish captures...' pattern?
- Information organisation: Is the response well-structured and easy to follow?
- Appropriate length: Is the response neither too brief nor too verbose?

**4. Cultural sensitivity**
- Respectful representation: Does it portray African cultures respectfully?
- Diversity awareness: Does it acknowledge the diversity within African regions?
- Avoiding stereotypes: Does it avoid oversimplification or cultural clichés?

When adapting this framework for other domains (e.g., history, science, agriculture), replace cultural sensitivity with domain-appropriate ethical or contextual criteria, if necessary.

#### Evaluation tips for conversational models

1. **Focus on helpfulness**: Would this response be useful to someone planning to visit the region?
2. **Check consistency**: Do similar questions get appropriately similar responses?
3. **Assess engagement**: Does the response encourage further conversation or exploration?
4. **Verify cultural knowledge**: Cross-check food and cultural claims against reliable sources when possible.
5. **Consider user intent**: Does the response match what the user was likely seeking?

#### Adapting evaluation to your domain

**Important**: Adjust the evaluation criteria based on your specific conversational model:
- **Domain expertise**: Replace food/culture criteria with your domain-specific accuracy measures.
- **Response style**: Adapt tone and format expectations to match your training data.
- **Success metrics**: Define what constitutes a "good" response for your use case.
- **Cultural considerations**: Ensure evaluation criteria respect the cultural context of your domain.
- **User expectations**: Consider what your target users would find most valuable.

This qualitative evaluation process helps you understand your model's conversational abilities and identify areas for improvement in future training iterations.


You have completed the technical implementation and evaluation of your conversational model. Through human evaluation and spot-checking, you have assessed how effectively your fine-tuned model responds to user prompts and maintains context, tone, and accuracy. The next step is to reflect on the development process and document key insights to guide future improvements and iterations.

## Activity 5: Iteration

One of the key parts of building any machine learning system is experimenting with different hyperparameters, datasets, models, or other variations of the pipeline that you just implemented. Even the most experienced machine learning researchers and engineers rarely get all of these things right without systematic experiments.

If you identified any systematic issues or are otherwise not yet satisfied with the performance of your conversational model, think how you may be able to improve your conversational model. You can then go back to individual steps, make changes and repeat the training and evaluation of your model to determine whether the changes are effective.

To do these iterations most systematically, it is best if you do not change too many things at once, so that you learn which components affect your conversational model the most. Furthermore, make sure to keep a log of your experiments somewhere in which you detail the settings of each training run, so that you know at the end what worked best. That way, you can progressively optimize your machine learning model until you are satisfied with its performance.

## Activity 6: Reflection


------
> 💻 **Your task:**
>
> **Technical and project reflection:**
>
> Document your learning and plan improvements for your model development.
>
> - **What worked well:** Reflect on successful aspects of your conversational model. Which responses were most natural and helpful?
>
> - **Challenges encountered:** What difficulties did you face in creating conversational responses? How did you address issues with cultural sensitivity or factual accuracy?
>
> - **Data quality observations:** How effective was your synthetic generation (if any)? What would you improve about the prompt templates or response formatting?
>
> - **Model performance:** How well did your model engage users? What are its conversational strengths and weaknesses?
>
> - **Conversational AI insights:** What did you learn about building conversational models? Which types of questions were hardest for your model to answer naturally?
>
> ------
> **Learning journey reflection:**
>
> Take a few minutes to think about your own learning journey in conversational AI development. Use these prompts to guide a short written reflection.
>
>  - **What aspects of the course helped you understand model development and LoRA fine-tuning most effectively?** Which techniques for creating conversational data were most valuable?
>
>  - **How did your thinking about conversational AI workflows change as you moved from data creation to fine-tuning and evaluation?**
>
>  - **Looking beyond this specific project, how will you apply these conversational AI skills when building models for other domains or cultural contexts in the future?**
>
>  - **What ethical considerations are especially important when building models?** How will you ensure responsible development of conversational AI systems?
>>
------

### Future improvements

List 3-5 ways to improve your model's conversational abilities

-
-
-
-
-


### Course connections

_Which course concepts were most important for this project?_

## Summary

Congratulations, you have completed the conversation capstone project!

This capstone demonstrated a complete machine learning workflow from conception to evaluation, highlighting the importance of careful data curation, parameter-efficient training methods, and evaluation practices in building reliable AI systems.